In [137]:
import polars as pl
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.iv import compare
from linearmodels.panel.results import compare
import statsmodels.api as sm
from statsmodels.api import qqplot
from linearmodels.panel import PanelOLS
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats as scipy_stats

import pyreadstat as prs
import pygwalker as pyg

import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

import json
pl.Config.set_tbl_rows(100)

from tqdm.notebook import tqdm
from itables import init_notebook_mode
# init_notebook_mode()

In [138]:
job_segmentation = pl.read_excel('../data/job_segmentation.xlsx')
job_segmentation

Industry,Sector (Primary / Secondary),is_secondary
str,str,i64
"""Ecology and Environmental Prot…","""Primary (often government or p…",0
"""Real Estate Operations""","""Secondary (commission-based, u…",1
"""Public Organizations, Council""","""Primary (government/public adm…",0
"""Mass Media, Publishing, Printi…","""Secondary (many precarious, fr…",1
"""Advertising and Marketing""","""Secondary (high turnover, proj…",1
"""Information Technology""","""Primary (high skill, good pay,…",0
"""Government and Public Administ…","""Primary (job security, pension…",0
"""Finances""","""Primary (core roles have high …",0
"""Social Services""","""Secondary (often low pay, high…",1


In [161]:
panel = pl.read_parquet('../data/wages_working_data2_1.parquet')
# panel = panel.filter(
#     pl.col('total_income') < 100_000
# )
panel = (
    panel
    .with_columns(gender = pl.when(pl.col('gender') == 'MALE').then(1).otherwise(0))
    .with_columns(recieves_pension = pl.when(pl.col('recieves_pension') == 'Yes').then(1).otherwise(0))
    .with_columns(recieves_pension_30days = pl.when(pl.col('recieves_pension_30days') == 'Yes').then(1).otherwise(0))
    .with_columns(income_decrease_bc_covid = pl.when(pl.col('income_decrease_bc_covid') == 'Yes').then(1).otherwise(0))

    .with_columns(harmfull_job = pl.when(pl.col('harmfull_job') == 'Yes').then(1).otherwise(0))
        
    .with_columns(pl.col('disability_class').str.replace(' ', '_').str.to_lowercase())
    .to_dummies('disability_class', drop_nulls=True, drop_first=True)
    .with_columns(has_disability = pl.when(pl.col('has_disability') == 'Yes').then(1).otherwise(0))
    .with_columns(male_retiered = pl.when(
        (pl.col('gender') == 1) & (pl.col('age') >= 65)
    ).then(1).otherwise(0))
    .with_columns(female_retiered = pl.when(
        (pl.col('gender') == 0) & (pl.col('age') >= 60)
    ).then(1).otherwise(0))
    .with_columns(is_retired = pl.col('male_retiered') + pl.col('female_retiered'))
    .to_dummies('educ_level', drop_nulls=True).drop('educ_level_common')
    .with_columns(has_disability_period = pl.col('has_disability') + pl.col('period'))
    .with_columns(has_disability_common_period = pl.col('has_disability_common') + pl.col('period'))
    .join(job_segmentation['Industry', 'is_secondary'], how='left', left_on='job_industry', right_on='Industry', validate='m:1')
    .with_columns(pl.col('is_secondary').fill_null(1))
    .filter(pl.col('is_secondary') == 1)
    .sort(['idind', 'year'])
    .to_pandas()
)
panel

,idind,year,recieves_pension,recieves_pension_30days,amount_pension,iea_income,total_income,amount_unemp_benefits,wage_j1,wage_j2,...,period_relevance,is_town,is_female,has_disability_common,male_retiered,female_retiered,is_retired,has_disability_period,has_disability_common_period,is_secondary
0,29.0,2017.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,-3,1,1,0,0,0,0,0,0,1
1,29.0,2020.0,0,0,0.0,0.0,43000.0,0.0,43000.0,0.0,...,0,1,1,0,0,0,0,1,1,1
2,29.0,2021.0,1,1,11300.0,0.0,0.0,0.0,0.0,0.0,...,1,1,1,0,0,0,0,0,0,1
3,30.0,2019.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1,1,0,0,0,0,0,0,0,1
4,30.0,2021.0,0,0,0.0,0.0,0.0,0.0,100000.0,0.0,...,1,1,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30923,60470.0,2020.0,0,0,0.0,0.0,55000.0,0.0,55000.0,0.0,...,0,1,0,0,0,0,0,1,1,1
30924,60473.0,2020.0,0,0,0.0,0.0,40000.0,0.0,40000.0,0.0,...,0,1,1,0,0,0,0,1,1,1
30925,60473.0,2021.0,0,0,0.0,0.0,55000.0,0.0,35000.0,0.0,...,1,1,1,0,0,0,0,0,0,1
30926,60474.0,2020.0,1,1,15600.0,0.0,15600.0,0.0,0.0,0.0,...,0,1,0,0,0,0,0,1,1,1


In [162]:
# panel.to_parquet('../data/wages_working_data3_1.parquet', index=False)

In [163]:
# (
#     panel
#     .filter(pl.col('has_disability') == 1)
#     ['job_industry'].value_counts().sort('count')#.drop_nulls()
# )

In [164]:
panel = panel[panel['year'].isin([
    2016,
    2017, 
    2018,
    2019, 
    2020, 
    # 2022
])]
panel

,idind,year,recieves_pension,recieves_pension_30days,amount_pension,iea_income,total_income,amount_unemp_benefits,wage_j1,wage_j2,...,period_relevance,is_town,is_female,has_disability_common,male_retiered,female_retiered,is_retired,has_disability_period,has_disability_common_period,is_secondary
0,29.0,2017.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,-3,1,1,0,0,0,0,0,0,1
1,29.0,2020.0,0,0,0.0,0.0,43000.0,0.0,43000.0,0.0,...,0,1,1,0,0,0,0,1,1,1
3,30.0,2019.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1,1,0,0,0,0,0,0,0,1
5,37.0,2020.0,1,1,14200.0,0.0,32100.0,0.0,17800.0,0.0,...,0,1,0,0,0,0,0,1,1,1
6,38.0,2016.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,...,-4,1,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30919,60467.0,2020.0,0,0,0.0,0.0,26600.0,0.0,26000.0,0.0,...,0,1,1,0,0,0,0,1,1,0
30922,60469.0,2020.0,1,1,9700.0,0.0,36700.0,0.0,27000.0,0.0,...,0,1,1,0,0,0,0,1,1,0
30923,60470.0,2020.0,0,0,0.0,0.0,55000.0,0.0,55000.0,0.0,...,0,1,0,0,0,0,0,1,1,1
30924,60473.0,2020.0,0,0,0.0,0.0,40000.0,0.0,40000.0,0.0,...,0,1,1,0,0,0,0,1,1,1


In [165]:
panel_2019_2020 = panel.copy()
panel_2019_2020['year'] = pd.to_datetime(panel_2019_2020['year'], format='%Y')
panel_2019_2020_ids = panel_2019_2020['idind']
panel_2019_2020 = panel_2019_2020.set_index(['idind', 'year'])
panel_2019_2020

recieves_pension  recieves_pension_30days  amount_pension  \
idind   year                                                                    
29.0    2017-01-01                 0                        0             0.0   
        2020-01-01                 0                        0             0.0   
30.0    2019-01-01                 0                        0             0.0   
37.0    2020-01-01                 1                        1         14200.0   
38.0    2016-01-01                 0                        0             0.0   
...                              ...                      ...             ...   
60467.0 2020-01-01                 0                        0             0.0   
60469.0 2020-01-01                 1                        1          9700.0   
60470.0 2020-01-01                 0                        0             0.0   
60473.0 2020-01-01                 0                        0             0.0   
60474.0 2020-01-01                 1                        1         15600.0   

                    iea_income  total_income  amount_unemp_benefits  wage_j1  \
idind   year                                                                   
29.0    2017-01-01         0.0           0.0                    0.0      0.0   
        2020-01-01         0.0       43000.0                    0.0  43000.0   
30.0    2019-01-01         0.0           0.0                    0.0      0.0   
37.0    2020-01-01         0.0       32100.0                    0.0  17800.0   
38.0    2016-01-01         0.0           0.0                    0.0      0.0   
...                        ...           ...                    ...      ...   
60467.0 2020-01-01         0.0       26600.0                    0.0  26000.0   
60469.0 2020-01-01         0.0       36700.0                    0.0  27000.0   
60470.0 2020-01-01         0.0       55000.0                    0.0  55000.0   
60473.0 2020-01-01         0.0       40000.0                    0.0  40000.0   
60474.0 2020-01-01         0.0       15600.0                    0.0      0.0   

                    wage_j2  income_decrease_bc_covid  \
idind   year                                            
29.0    2017-01-01      0.0                         0   
        2020-01-01      0.0                         1   
30.0    2019-01-01      0.0                         0   
37.0    2020-01-01      0.0                         0   
38.0    2016-01-01      0.0                         0   
...                     ...                       ...   
60467.0 2020-01-01      0.0                         0   
60469.0 2020-01-01      0.0                         0   
60470.0 2020-01-01      0.0                         0   
60473.0 2020-01-01      0.0                         1   
60474.0 2020-01-01      0.0                         0   

                                                          work_status  ...  \
idind   year                                                           ...   
29.0    2017-01-01                                You are not working  ...   
        2020-01-01                          You are currently working  ...   
30.0    2019-01-01                          You are currently working  ...   
37.0    2020-01-01                          You are currently working  ...   
38.0    2016-01-01                                You are not working  ...   
...                                                               ...  ...   
60467.0 2020-01-01  You are on paid leave: maternity leave or taki...  ...   
60469.0 2020-01-01                          You are currently working  ...   
60470.0 2020-01-01                          You are currently working  ...   
60473.0 2020-01-01                          You are currently working  ...   
60474.0 2020-01-01                          You are currently working  ...   

                    period_relevance  is_town  is_female  \
idind   year                                               
29.0    2017-01-01                -3        1   

In [166]:
model = smf.ols('wages ~ has_disablity + period + I(has_disablity * period)', data=panel_2019_2020).fit(cov_type='cluster', cov_kwds={'groups': panel_2019_2020_ids})
print(model.summary2())

                          Results: Ordinary least squares
Model:                   OLS                   Adj. R-squared:          0.002      
Dependent Variable:      wages                 AIC:                     392907.8646
Date:                    2026-05-12 10:30      BIC:                     392938.8753
No. Observations:        17200                 Log-Likelihood:          -1.9645e+05
Df Model:                3                     F-statistic:             15.36      
Df Residuals:            17196                 Prob (F-statistic):      5.86e-10   
R-squared:               0.002                 Scale:                   4.8778e+08 
-----------------------------------------------------------------------------------
                            Coef.     Std.Err.    z    P>|z|    [0.025     0.975]  
-----------------------------------------------------------------------------------
Intercept                 26599.0532  283.5295 93.8140 0.0000 26043.3454 27154.7609
has_disablity     

In [167]:
model1 = PanelOLS.from_formula('wages ~ I(has_disablity * period) + EntityEffects + TimeEffects', data=panel_2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model1.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  wages   R-squared:                        0.0003
Estimator:                   PanelOLS   R-squared (Between):             -0.0008
No. Observations:               17200   R-squared (Within):              -0.0004
Date:                Tue, May 12 2026   R-squared (Overall):             -0.0007
Time:                        10:30:14   Log-likelihood                -1.844e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2.8163
Entities:                        6329   P-value                           0.0933
Avg Obs:                       2.7176   Distribution:                 F(1,10866)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             1.8772
                            

In [168]:
model2 = PanelOLS.from_formula('wages ~ I(has_disablity * period) + EntityEffects + TimeEffects', data=panel_2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model2.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  wages   R-squared:                        0.0003
Estimator:                   PanelOLS   R-squared (Between):             -0.0008
No. Observations:               17200   R-squared (Within):              -0.0004
Date:                Tue, May 12 2026   R-squared (Overall):             -0.0007
Time:                        10:30:20   Log-likelihood                -1.844e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      2.8163
Entities:                        6329   P-value                           0.0933
Avg Obs:                       2.7176   Distribution:                 F(1,10866)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             1.8772
                            

In [169]:
2.2e-16

2.2e-16

In [171]:
model3 = PanelOLS.from_formula('wages ~ has_disability_period + EntityEffects + TimeEffects', drop_absorbed=True, data=panel_2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model3.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  wages   R-squared:                        0.0007
Estimator:                   PanelOLS   R-squared (Between):             -0.1232
No. Observations:               17200   R-squared (Within):              -0.1137
Date:                Tue, May 12 2026   R-squared (Overall):             -0.1013
Time:                        10:30:33   Log-likelihood                -1.844e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      8.0502
Entities:                        6329   P-value                           0.0046
Avg Obs:                       2.7176   Distribution:                 F(1,10866)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             9.0347
                            

In [156]:
-1.208e+04

-12080.0

In [ ]:
model4 = PanelOLS.from_formula('wages ~ has_disability_period + EntityEffects + TimeEffects', drop_absorbed=True, data=panel_2019_2020).fit(cov_type='clustered', cluster_entity=True, group_debias=False)
print(model4.summary)